In [4]:
import os
import boto3
import logging
from botocore.exceptions import NoCredentialsError, ClientError

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def ensure_bucket_exists(s3_client, bucket_name):
    """
    Checks if the S3 bucket exists and creates it if it does not.
    This version is simplified for generic S3 compatibility.

    Args:
        s3_client: The initialized Boto3 S3 client configured with the endpoint URL.
        bucket_name (str): The name of the S3 bucket.
    
    Returns:
        bool: True if the bucket exists or was successfully created, False otherwise.
    """
    try:
        # Check if the bucket exists using head_bucket
        s3_client.head_bucket(Bucket=bucket_name)
        logging.info(f"Bucket '{bucket_name}' already exists.")
        return True
    except ClientError as e:
        error_code = e.response['Error']['Code']
        
        # 404 means the bucket does not exist
        if error_code == '404':
            logging.info(f"Bucket '{bucket_name}' does not exist. Attempting creation...")
            
            try:
                # Simplified bucket creation for generic S3 services
                s3_client.create_bucket(Bucket=bucket_name)
                
                logging.info(f"Bucket '{bucket_name}' successfully created.")
                return True
            except ClientError as create_e:
                 # Handle specific errors during creation (e.g., name conflict, permission)
                logging.error(f"Failed to create bucket '{bucket_name}': {create_e}")
                return False
        
        # 403 means Forbidden (Permission Denied) or other access issue
        elif error_code in ['403', 'Forbidden']:
            logging.error(f"Access denied. Cannot check or create bucket '{bucket_name}'. Check your keys and permissions.")
            return False
        
        else:
            logging.error(f"Error checking bucket existence for '{bucket_name}': {e}")
            return False
    except Exception as e:
        logging.error(f"An unexpected error occurred during bucket check/creation: {e}")
        return False

def upload_directory_to_s3(local_directory, bucket_name, s3_prefix='', endpoint_url=None, aws_access_key_id=None, aws_secret_access_key=None):
    """
    Recursively uploads files from a local directory to an S3-compatible bucket,
    maintaining the directory structure, and ensuring the bucket exists first.

    Args:
        local_directory (str): The path to the local directory to upload.
        bucket_name (str): The name of the S3 bucket.
        s3_prefix (str): An optional S3 prefix (folder path) to upload files into.
        endpoint_url (str, optional): The custom endpoint URL for the S3 provider.
        aws_access_key_id (str, optional): The access key ID.
        aws_secret_access_key (str, optional): The secret access key.
    """
    # Normalize paths
    local_directory = os.path.abspath(local_directory)
    if not os.path.isdir(local_directory):
        logging.error(f"Local directory not found: {local_directory}")
        return

    # Ensure s3_prefix ends with a slash if it's not empty, unless it's the root.
    if s3_prefix and not s3_prefix.endswith('/'):
        s3_prefix += '/'

    try:
        # Initialize S3 client using explicit configuration for generic S3 compatibility
        s3_client = boto3.client(
            's3', 
            endpoint_url=endpoint_url,
            # Boto3 still expects these parameter names, even for generic S3
            aws_access_key_id=aws_access_key_id,
            aws_secret_access_key=aws_secret_access_key,
            # A region is still required by boto3, but for generic S3, the value often doesn't matter
            region_name='us-east-1' 
        )
        
        # --- Check and create bucket if necessary ---
        if not ensure_bucket_exists(s3_client, bucket_name):
            logging.error("Failed to ensure bucket exists. Aborting upload.")
            return
        
        logging.info(f"Starting upload from '{local_directory}' to s3://{bucket_name}/{s3_prefix}")
        
        # Walk through the local directory structure
        for root, dirs, files in os.walk(local_directory):
            # Calculate the relative path from the root directory passed to the function
            relative_path = os.path.relpath(root, local_directory)

            for file_name in files:
                local_file_path = os.path.join(root, file_name)

                # Construct the S3 key
                if relative_path == ".":
                    # If we are in the root of the source directory, don't include '.'
                    s3_key = s3_prefix + file_name
                else:
                    # Replace OS path separator (\ on Windows) with S3 forward slash
                    s3_key = s3_prefix + os.path.join(relative_path, file_name).replace(os.path.sep, '/')

                try:
                    # Upload the file
                    # The extra arguments (endpoint_url, etc.) are only passed during client initialization
                    s3_client.upload_file(local_file_path, bucket_name, s3_key)
                    logging.info(f"Successfully uploaded: {local_file_path} -> s3://{bucket_name}/{s3_key}")
                
                except ClientError as e:
                    logging.error(f"Error uploading {local_file_path}: {e}")
                
                except Exception as e:
                    logging.error(f"An unexpected error occurred during file upload: {e}")

        logging.info("Upload process completed.")

    except Exception as e:
        # Catch initialization errors if keys or endpoint are missing/invalid
        logging.error(f"Failed to initialize S3 client or connect: {e}")



In [5]:
if __name__ == "__main__":
    # --- CONFIGURATION (UPDATED FOR GENERIC S3) ---
    
    # The endpoint URL for your S3 compatible provider (e.g., MinIO, DigitalOcean Spaces, Wasabi)
    # The default is now set to the local Kubernetes service address.
    ENDPOINT_URL = "http://local-s3-service.ezdata-system.svc.cluster.local:30000" 

    # --- CREDENTIALS CONFIGURATION ---
    TOKEN_PATH = "/etc/secrets/ezua/.auth_token"
    
    # 1. Read ACCESS KEY from the specified file
    try:
        with open(TOKEN_PATH, 'r') as f:
            # Read content and strip any leading/trailing whitespace (including newlines)
            AWS_ACCESS_KEY_ID = f.read().strip()
            logging.info(f"Successfully read access key from {TOKEN_PATH}.")
    except FileNotFoundError:
        AWS_ACCESS_KEY_ID = None
        logging.error(f"Error: Authentication token file not found at {TOKEN_PATH}.")
    except Exception as e:
        AWS_ACCESS_KEY_ID = None
        logging.error(f"Error reading authentication token: {e}")

    # 2. Set SECRET KEY to the hardcoded value "s3"
    AWS_SECRET_ACCESS_KEY = "s3"
    
    # The local folder you want to upload. 
    # this is root of directory this is not create - but subdirectories are mapped to prefix
    SOURCE_DIR = "/mnt/models-pvc/models_dnld" 

    # The name of your S3 bucket.
    TARGET_BUCKET = "s3models" 
    
    # Optional: A prefix inside the S3 bucket. 
    #TARGET_PREFIX = "test-upload/" 
    TARGET_PREFIX=""

    # --- EXECUTION ---
    
    # Check if we have an access key before proceeding to create dummy data or upload
    if not AWS_ACCESS_KEY_ID:
        logging.error("Access key is missing. Cannot proceed with S3 operations.")
    else:
        # Create a dummy folder structure for testing if it doesn't exist
        if not os.path.exists(SOURCE_DIR):
            print(f"Creating dummy directory structure at '{SOURCE_DIR}' for demonstration.")
            os.makedirs(os.path.join(SOURCE_DIR, 'reports/2023'))
            with open(os.path.join(SOURCE_DIR, 'config.ini'), 'w') as f:
                f.write("[Settings]\nversion=1.0")
            with open(os.path.join(SOURCE_DIR, 'reports/2023/q4.txt'), 'w') as f:
                f.write("Q4 Report Data")
            print("Dummy structure created.")


        # Call the main upload function
        # WARNING: This will attempt to connect and upload. Ensure your configuration 
        # (ENDPOINT_URL, KEYS, and TARGET_BUCKET) is correct.
        # UNCOMMENT THE LINE BELOW TO EXECUTE THE UPLOAD AND BUCKET CREATION
        upload_directory_to_s3(
            SOURCE_DIR, 
            TARGET_BUCKET, 
            TARGET_PREFIX, 
            ENDPOINT_URL, 
            AWS_ACCESS_KEY_ID, 
            AWS_SECRET_ACCESS_KEY
        )
        
#         # For demonstration purposes, just show the paths that would be used:
#         print("\n--- Dry Run Path Mapping (Uncomment the function call above to execute) ---")
#         dry_run_directory = SOURCE_DIR
#         dry_run_bucket = TARGET_BUCKET
#         dry_run_prefix = TARGET_PREFIX
        
#         for root, dirs, files in os.walk(dry_run_directory):
#             relative_path = os.path.relpath(root, dry_run_directory)
            
#             for file_name in files:
#                 local_file_path = os.path.join(root, file_name)
                
#                 if relative_path == ".":
#                     s3_key = dry_run_prefix + file_name
#                 else:
#                     s3_key = dry_run_prefix + os.path.join(relative_path, file_name).replace(os.path.sep, '/')
                
#                 print(f"Local: {local_file_path.ljust(40)} -> S3 Key: {s3_key}")
                
#         print("\n-----------------------------------------------------------------------")
#         print("If executed, the script will first ensure the bucket exists on your generic S3 provider, then upload the files.")
#         print("Be sure to set ENDPOINT_URL, AWS_ACCESS_KEY_ID, and AWS_SECRET_ACCESS_KEY in the configuration block.")

Successfully read access key from /etc/secrets/ezua/.auth_token.
Bucket 's3models' does not exist. Attempting creation...
Bucket 's3models' successfully created.
Starting upload from '/mnt/models-pvc/models_dnld' to s3://s3models/
Successfully uploaded: /mnt/models-pvc/models_dnld/hub/.locks/models--google--codegemma-7b-it/0e10e77a6d85df8b59a7c36f25b1b4942754903b.lock -> s3://s3models/hub/.locks/models--google--codegemma-7b-it/0e10e77a6d85df8b59a7c36f25b1b4942754903b.lock
Successfully uploaded: /mnt/models-pvc/models_dnld/hub/.locks/models--google--codegemma-7b-it/10f648856f02b45d78200efaebd1d59d6cfe7ad0.lock -> s3://s3models/hub/.locks/models--google--codegemma-7b-it/10f648856f02b45d78200efaebd1d59d6cfe7ad0.lock
Successfully uploaded: /mnt/models-pvc/models_dnld/hub/.locks/models--google--codegemma-7b-it/147d809c91b08ad1c6bb6483a19eb2f1d83c665e.lock -> s3://s3models/hub/.locks/models--google--codegemma-7b-it/147d809c91b08ad1c6bb6483a19eb2f1d83c665e.lock
Successfully uploaded: /mnt/mod